# Chinese EV market intelligence: validation and screening

**Question:** How do selected Chinese EV models compare on like-for-like vehicle attributes, and where is international brand presence documented?

For analysts/importers building an initial competitor research shortlist. Source review: **2026-09-17**. The original eight-model, seven-brand selection is retained; it is not a representative census. This notebook validates curated CSV inputs and reproduces the dashboard definitions. It does not collect live data or make market-entry recommendations.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Supported launch directories: repository root or notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'dashboard' / 'data.py').is_file():
    raise RuntimeError('Run this notebook from the repository root or notebooks directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from dashboard.data import load_data, comparable_models, kpis, presence_matrix
models, brands, presence = load_data()
print('Validated model / brand / presence rows:', len(models), len(brands), len(presence))

Validated model / brand / presence rows: 8 7 28


## Sources and verification

See [source decisions](../data/SOURCES.md) and [field definitions](../data/README.md). Five EV Database listings support European specifications and Dutch prices. The Australian JAECOO MY2026 sheet supports 402 km WLTP, LFP and 58.9 kWh with an unspecified capacity basis. Xiaomi's March 2024 company disclosure supports a historical up-to 800 km CLTC claim. Li Auto's profile supports EREV identity but not the unresolved L9 trim's numerical specifications.

Source-checked means matched to a publication, not independent road testing. Review dates are not model years. Unsupported numbers remain blank.

In [2]:
display(models[['brand', 'model', 'variant_context', 'source', 'source_url', 'verification_date', 'verification_status']])

,brand,model,variant_context,source,source_url,verification_date,verification_status
0,BYD,Seal 82.5 kWh AWD Excellence,MY23-25,EV Database,https://ev-database.org/car/2002/BYD-SEAL-825-...,2026-09-17,verified_values
1,BYD,Dolphin 60.4 kWh,MY25,EV Database,https://ev-database.org/car/3297/BYD-DOLPHIN-6...,2026-09-17,verified_values
2,Zeekr,001 Long Range RWD,MY24,EV Database,https://ev-database.org/car/1933/Zeekr-001-Lon...,2026-09-17,verified_values
3,XPeng,G6 RWD Long Range,MY25,EV Database,https://ev-database.org/car/3275/XPENG-G6-RWD-...,2026-09-17,verified_values
4,NIO,ET5 Touring Long Range,Long Range; listed since 2023,EV Database,https://ev-database.org/car/1916/NIO-ET5-Touri...,2026-09-17,verified_values
5,Li Auto,L9,Original L9 selection; exact model year unreso...,Li Auto investor relations,https://ir.lixiang.com/,2026-09-17,partial
6,Xiaomi Auto,SU7 Max,Historical launch specification; March 2024 di...,Xiaomi FY2023 results announcement; p.4,https://ir.mi.com/system/files-encrypted/nasda...,2026-09-17,partial
7,Jaecoo,J5 EV,MY2026; specification sheet edition 1.01 April...,Omoda Jaecoo Australia specification sheet; p.5,https://www.omodajaecoo.com.au/sites/default/f...,2026-09-17,verified_values


## Data quality

The shared loader validates required columns, unique keys, allowed categories, date syntax and metric context. Brand metadata joins many-to-one. Presence stays separate to prevent fourfold row multiplication. Missing values are neither zero nor a reason to impute another model's value.

In [3]:
numeric_fields = ['price_eur', 'electric_range_km', 'battery_capacity_kwh', 'total_range_km']
quality = pd.DataFrame({'non_missing': models[numeric_fields].notna().sum(),
                        'missing': models[numeric_fields].isna().sum()})
display(quality)
assert not models.duplicated(['brand', 'model']).any()
assert not brands.duplicated('brand').any()
assert not presence.duplicated(['brand', 'region']).any()
assert len(models) == 8 and len(brands) == 7 and len(presence) == 28
print('Unique keys and expected cardinality confirmed.')

,non_missing,missing
price_eur,5,3
electric_range_km,7,1
battery_capacity_kwh,6,2
total_range_km,0,8


Unique keys and expected cardinality confirmed.


## Semantic comparability and transformations

The repair separates powertrain from chemistry, electric from total range, and usable from unspecified capacity. The old mixed averages and subjective indices are retired. No price is imputed or converted from CNY.

Only BEVs with Dutch listed RRP, currently listed availability and EV Database Real Range qualify for the scatter. The discontinued Seal listing remains a historical reference in the table. CLTC and WLTP records retain their own standards. These rules live in `dashboard/data.py` and are reused by the app.

In [4]:
display(models[['brand', 'model', 'powertrain_type', 'electric_range_km', 'range_standard',
                'total_range_km', 'battery_chemistry', 'battery_capacity_kwh', 'battery_capacity_basis']])
summary = kpis(models)
display(pd.Series(summary, name='Dataset counts'))
comparison = comparable_models(models)
display(comparison[['brand', 'model', 'price_eur', 'price_market', 'electric_range_km', 'range_standard']])
assert summary == {'models': 8, 'brands': 7, 'comparable': 4, 'range': 7}

,brand,model,powertrain_type,electric_range_km,range_standard,total_range_km,battery_chemistry,battery_capacity_kwh,battery_capacity_basis
0,BYD,Seal 82.5 kWh AWD Excellence,BEV,445.0,EV Database Real Range,NaN,LFP,82.5,usable
1,BYD,Dolphin 60.4 kWh,BEV,350.0,EV Database Real Range,NaN,LFP,60.5,usable
2,Zeekr,001 Long Range RWD,BEV,505.0,EV Database Real Range,NaN,NMC811,94.0,usable
3,XPeng,G6 RWD Long Range,BEV,450.0,EV Database Real Range,NaN,LFP,80.0,usable
4,NIO,ET5 Touring Long Range,BEV,485.0,EV Database Real Range,NaN,NMC,90.0,usable
5,Li Auto,L9,EREV,NaN,NaN,NaN,NaN,NaN,NaN
6,Xiaomi Auto,SU7 Max,BEV,800.0,CLTC,NaN,NaN,NaN,NaN
7,Jaecoo,J5 EV,BEV,402.0,WLTP,NaN,LFP,58.9,unspecified


models        8
brands        7
comparable    4
range         7
Name: Dataset counts, dtype: int64

,brand,model,price_eur,price_market,electric_range_km,range_standard
1,BYD,Dolphin 60.4 kWh,37190.0,Netherlands,350.0,EV Database Real Range
2,Zeekr,001 Long Range RWD,55990.0,Netherlands,505.0,EV Database Real Range
3,XPeng,G6 RWD Long Range,48990.0,Netherlands,450.0,EV Database Real Range
4,NIO,ET5 Touring Long Range,72900.0,Netherlands,485.0,EV Database Real Range


## Descriptive price/range comparison

Denominator: four qualifying model rows out of eight selected. Prices are source-listed Dutch RRPs excluding indirect incentives. Segment, options, battery purchase terms and model-year differences still matter. No price/range ratio is interpreted as cost per kilometre.

In [5]:
fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
ax.scatter(comparison.price_eur, comparison.electric_range_km, color='#087f8c', s=65)
for row in comparison.itertuples():
    ax.annotate(f'{row.brand} {row.model.split()[0]}', (row.price_eur, row.electric_range_km),
                xytext=(6, 6), textcoords='offset points')
ax.set(xlabel='Netherlands listed RRP (€)', ylabel='EV Database Real Range (km)')
ax.margins(x=.2, y=.25)
ax.grid(alpha=.15)
plt.show()
plt.close(fig)

/var/folders/7h/cy7wds_s1gqcmjdfc8c48w_h0000gn/T/ipykernel_56279/91862903.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
x = comparison.loc[comparison.brand.eq('XPeng')].iloc[0]
z = comparison.loc[comparison.brand.eq('Zeekr')].iloc[0]
print(f'Observation: Zeekr minus XPeng = {z.electric_range_km - x.electric_range_km:.0f} km and €{z.price_eur - x.price_eur:,.0f}.')
print('Possible business relevance: investigate the price/range trade-off.')
print('Limitation: different vehicle formats and equipment; no best-buy or sales-performance conclusion.')

Observation: Zeekr minus XPeng = 55 km and €7,000.
Possible business relevance: investigate the price/range trade-off.
Limitation: different vehicle formats and equipment; no best-buy or sales-performance conclusion.


## Documented international presence

A resolved cell needs a named-country commercial listing, official sales/test-drive channel or delivery report. This is brand-level evidence, potentially for another model/powertrain. A country witness is not region-wide penetration. Historical delivery evidence is dated, not a guarantee of continuous operations.

Unknown means no qualifying evidence retained in this bounded review. It never means absent. Research coverage is uneven; the count of documented regions must not rank brands or assess market attractiveness.

In [7]:
display(presence_matrix(presence))
resolved = presence.presence_status.ne('unknown')
print(f'Evidence coverage: {resolved.sum()} / {len(presence)} brand–region cells resolved.')
display(presence.loc[resolved, ['brand', 'region', 'evidence_country', 'evidence_date', 'source_url', 'evidence_note']])
assert resolved.sum() == 7
assert presence.presence_status.eq('unknown').sum() == 21

region,Europe,Southeast Asia,Middle East,Latin America
brand,,,,
BYD,present,present,unknown,present
Jaecoo,present,unknown,unknown,unknown
Li Auto,unknown,unknown,unknown,unknown
NIO,present,unknown,unknown,unknown
XPeng,present,unknown,unknown,unknown
Xiaomi Auto,unknown,unknown,unknown,unknown
Zeekr,present,unknown,unknown,unknown


Evidence coverage: 7 / 28 brand–region cells resolved.


,brand,region,evidence_country,evidence_date,source_url,evidence_note
0,BYD,Europe,Netherlands,2026-09-17,https://ev-database.org/car/3297/BYD-DOLPHIN-6...,Dutch price and open-ended available-to-order ...
1,BYD,Southeast Asia,Thailand,2026-09-17,https://www.byd.com/en-th,Official country catalogue offers passenger mo...
3,BYD,Latin America,Brazil,2026-09-17,https://www.byd.com/br/termos-de-uso,Official country site states that vehicles are...
4,Jaecoo,Europe,United Kingdom,2026-07-17,https://jaecoo.co.uk/news/flagship-jaecoo-8-sh...,Official report of UK customer deliveries. Bra...
12,NIO,Europe,Netherlands,2026-09-17,https://ev-database.org/car/1916/NIO-ET5-Touri...,Dutch price and open-ended available-to-order ...
16,XPeng,Europe,Netherlands,2026-09-17,https://ev-database.org/car/3275/XPENG-G6-RWD-...,Dutch price and open-ended available-to-order ...
24,Zeekr,Europe,Netherlands,2026-09-17,https://ev-database.org/car/1933/Zeekr-001-Lon...,Dutch price and open-ended available-to-order ...


## Interpretation and limitations

**Observation:** Five selected brands have European country evidence; BYD also has Thailand and Brazil evidence in this file. **Possible business relevance:** locate competitor evidence and prioritize missing-source research. **Limitation:** incomplete, unequal evidence coverage prevents a regional strength ranking.

The case cannot establish sales, market share, demand, profitability, causality or entry feasibility. It is a small curated selection with changing external sources and historical variants. Li Auto lacks verified trim-level numbers; Xiaomi lacks a comparable EUR price and retained battery specifications. Source reuse rights are not asserted as an open-data licence.

The original project described manual curation. Codex assisted this repair, documentation and checks. Source publications—not AI output—support the factual values. This notebook does not modify the inputs.